In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# EU Tech Job Market
## What happened to tech jobs in Europe since 2020, and what do they pay?

### Abstract

_TODO: write at the end, when the results are ready._

### Table of Contents

* [1. Problem formulation](#chapter1)
* [2. Data sources](#chapter2)
* [3. Validation and cleaning](#chapter3)
* [4. Consolidation](#chapter4)
* [5. Statistical methods](#chapter5)
* [6. Demand for tech jobs](#chapter6)
* [7. Roles and languages](#chapter7)
* [8. Pay, shortage and remote work](#chapter8)
* [9. Comparison with a forecast](#chapter9)
* [10. Limitations](#chapter10)
* [11. Conclusion](#chapter11)
* [12. References](#chapter12)

### 1. Problem formulation <a class="anchor" id="chapter1"></a>

Tech jobs are often called "the jobs of the future". But since 2022 I have also read many news stories about layoffs and about fewer job ads for developers. Both stories may be true at the same time if they measure different things. This matters for everyone who works in IT or wants to start: it affects which skills to learn, where to look for work and what salary to expect.

The main question of this project is:

> **What happened to the tech job market in Europe between 2020 and 2026, and how does it look today?**

#### 1.1. Why one data source is not enough <a class="anchor" id="section_1_1"></a>

The job market has several sides, and each public source shows only one of them:

| View | Source | What it shows | What it cannot show |
|---|---|---|---|
| Demand | Indeed job postings [1] | How the number of job ads changes, every day | How many people work in IT; salaries |
| Supply | Eurostat employment statistics [3] | How many ICT specialists work in each country | Which roles; pay; only one value per year |
| Shortage | Eurostat enterprise survey [4] | Share of firms that could not fill ICT jobs | Which roles; only every second year since 2020 |
| People | Stack Overflow Developer Survey [2] | Roles, salaries, remote work, languages, AI use | Only people who answered; not a random sample |
| Skills | GitHub Innovation Graph [7] | Which programming languages people push code in | Anything about jobs or pay |

Each of these organizations publishes analyses based only on its own data: Indeed Hiring Lab writes about job postings [1], Eurostat about employment [3], [4] and Stack Overflow about its survey [2]. In this project I put the sources together. The goal is to see if they tell the same story, and to explain it when they do not.

#### 1.2. Research questions <a class="anchor" id="section_1_2"></a>

| # | Question | Data | Method |
|---|---|---|---|
| Q1 | How did job postings for tech roles change since 2020, compared with all job postings? | Indeed (DE, FR, GB) | Index comparison (descriptive) |
| Q2 | Does the number of people in ICT jobs move in the same direction as the job ads? | Eurostat + Indeed (DE, FR) | Merge by country and year (descriptive) |
| Q3 | Did the mix of developer roles change between 2023 and 2026? | Stack Overflow | Chi-square test, **H3** |
| Q4 | Do some role groups earn more than others, after adjusting for price levels? | Stack Overflow + Eurostat | Kruskal–Wallis test, **H4** |
| Q5 | Did remote work become less common between 2022 and 2024? | Stack Overflow | Chi-square test, **H5** |
| Q6 | Do countries with a stronger ICT shortage pay developers more? | Eurostat + Stack Overflow | Spearman correlation, **H6** |
| Q7 | Do GitHub and the survey agree on the most used languages? | GitHub + Stack Overflow | Spearman correlation (exploratory) |
| Q8 | Do the findings agree with a published forecast for 2030? | WEF Future of Jobs Report 2025 [8] | Comparison (qualitative) |

Q1, Q2, Q7 and Q8 describe the data. Q3–Q6 are formal tests with hypotheses that I fixed in advance (section 1.3).

#### 1.3. Hypotheses and rules fixed before the analysis <a class="anchor" id="section_1_3"></a>

If we look at the data first and choose the test later, it is easy to find a "significant" result by chance. This is called p-hacking. To avoid it, I wrote down all tests and rules below **before** running any analysis. This text was committed to Git before the analysis code, so the commit history shows the order.

**H3 (roles).** Let $p_g^{(t)}$ be the share of respondents in role group $g$ in survey year $t$:

$$H_0: \; p_g^{(2023)} = p_g^{(2026)} \;\text{ for every group } g \qquad H_1: \; \text{at least one share is different} \tag{1}$$

**H4 (pay).** Let $F_g$ be the distribution of real yearly pay (formula (4) below) in role group $g$, in the 2025 survey:

$$H_0: \; F_1 = F_2 = \dots = F_k \qquad H_1: \; \text{at least one group tends to earn more or less} \tag{2}$$

**H5 (remote work).** The shares of *Remote*, *Hybrid* and *In-person* answers are the same in 2022 and in 2024 ($H_0$), or they are different ($H_1$). In 2022 two of the answers had other names ("Fully remote", "Full in-person"), but the meaning is the same.

**H6 (shortage and pay).** Let $\rho_s$ be the Spearman correlation across countries between the share of firms with hard-to-fill ICT vacancies (Eurostat, 2024) and the median real pay of developers (Stack Overflow, 2024):

$$H_0: \; \rho_s = 0 \qquad H_1: \; \rho_s \neq 0 \tag{3}$$

With the 2024 data, 24 countries meet both conditions (at least 30 pay answers and a Eurostat value). With so few points, only a strong correlation ($|\rho_s|$ above about 0.40) can be significant. So a non-significant result will mean "no clear evidence", not "no relation".

**Rules for all tests:**

* Significance level $\alpha = 0.05$. H3–H6 are four tests, so the p-values are corrected with the Holm method [9].
* Every test also reports an effect size, because with thousands of answers even a very small difference can be "significant".
* A group or a country takes part in a test only if it has **at least 30 answers**.
* Bootstrap confidence intervals use 10 000 resamples with a fixed random seed (42), so the numbers are the same on every run.
* Role groups: *Software development*, *Data & AI*, *Cloud & operations*, *Security*, *Architecture & management*, *Other*. Students and retired people are not counted. The exact mapping of every survey answer is shown in section 4.

**Rules for pay (H4 and H6):**

* Only employees are counted: "Employed, full-time" in 2022–2024 and "Employed" in 2025–2026 (from 2025 the survey no longer separates full-time and part-time). Freelancers are left out, because their income is not comparable with a salary.
* Answers without pay or with pay of 0 are removed. After that, the lowest 1% and the highest 1% of each survey year are removed as likely input errors.
* Pay is made comparable between countries with the Eurostat price level index $PLI$ (EU27 = 100) for actual individual consumption:

$$w_{\text{real}} = \frac{w_{\text{USD}} \,/\, r_t}{PLI_{c,t} \,/\, 100} \tag{4}$$

Where:
* $w_{\text{USD}}$ is the yearly pay in US dollars from the survey (`ConvertedCompYearly`),
* $r_t$ is the average number of US dollars for 1 euro in year $t$ [6],
* $PLI_{c,t}$ is the price level index of country $c$ in year $t$ [5]. A value of 60 means that prices are 40% lower than the EU average.

The 2026 survey is used with the 2025 values of $r_t$ and $PLI$, because 2026 values are not published yet.

#### 1.4. Scope and assumptions <a class="anchor" id="section_1_4"></a>

* **Europe** means 31 countries: the EU27, the United Kingdom, Switzerland, Norway and Iceland. Some sources cover only some of them; each section says which countries it uses.
* **Time:** from February 2020 (the start of the Indeed data) to November 2026. The survey years are 2022–2026, because older surveys ask about roles and remote work in a different way.
* **The Indeed numbers are an index, not counts.** Each series is a seasonally adjusted 7-day average that equals 100 on 1 February 2020. A value of 50 means half as many postings as on that day [1]. So we can compare *changes* between sectors, but not their sizes.
* **The sources cover different years.** Indeed goes to November 2026 and the survey to 2026. Employment and price levels end in 2025, and hard-to-fill vacancies exist only for 2020, 2022 and 2024. Each question uses only the years that its data has.
* **The survey gets smaller.** The number of European respondents fell from 38 050 in 2023 to 9 356 in 2026. Fewer answers do not mean fewer jobs. The group of people who answer can also change from year to year, so comparisons between years describe the respondents, not the whole market.
* **Stock and flow are different things.** Eurostat counts people who *have* an ICT job (stock). Indeed counts *open* jobs (flow). The two can move in different directions.
* **The survey is not a random sample** of all developers. People decide themselves if they answer, so the results describe the people who answered, not all developers.
* **This is not a forecast.** The data shows what happened until today. In section 9 I compare it with a published forecast, but I do not make my own predictions.

### 2. Data sources <a class="anchor" id="chapter2"></a>

_TODO_

### 3. Validation and cleaning <a class="anchor" id="chapter3"></a>

_TODO_

### 4. Consolidation <a class="anchor" id="chapter4"></a>

_TODO_

### 5. Statistical methods <a class="anchor" id="chapter5"></a>

_TODO_

### 6. Demand for tech jobs <a class="anchor" id="chapter6"></a>

_TODO_

### 7. Roles and languages <a class="anchor" id="chapter7"></a>

_TODO_

### 8. Pay, shortage and remote work <a class="anchor" id="chapter8"></a>

_TODO_

### 9. Comparison with a forecast <a class="anchor" id="chapter9"></a>

_TODO_

### 10. Limitations <a class="anchor" id="chapter10"></a>

_TODO_

### 11. Conclusion <a class="anchor" id="chapter11"></a>

_TODO_

### 12. References <a class="anchor" id="chapter12"></a>

1. Indeed Hiring Lab. *Indeed Job Postings Index*. https://github.com/hiring-lab/job_postings_tracker (data under CC BY 4.0, commit `d876002`).
2. Stack Overflow. *Annual Developer Survey*, 2022–2026. https://survey.stackoverflow.co/ (data under ODbL 1.0).
3. Eurostat. *ICT specialists in employment*, dataset `isoc_sks_itspt`. https://ec.europa.eu/eurostat/statistics-explained/index.php/ICT_specialists_in_employment
4. Eurostat. *ICT specialists – statistics on hard-to-fill vacancies in enterprises*, dataset `isoc_ske_itrcrn2`. https://ec.europa.eu/eurostat/statistics-explained/index.php/ICT_specialists_-_statistics_on_hard-to-fill_vacancies_in_enterprises
5. Eurostat. *Purchasing power parities, price level indices, nominal and real expenditures (COICOP 2018)*, dataset `prc_ppp_ind_1`.
6. Eurostat. *Euro/ECU exchange rates – annual data*, dataset `ert_bil_eur_a`.
7. GitHub. *Innovation Graph*. https://github.com/github/innovationgraph (data under CC0 1.0, commit `db6ccdf`).
8. World Economic Forum (2025). *Future of Jobs Report 2025*. Insight Report, January 2025. https://www3.weforum.org/docs/WEF_Future_of_Jobs_Report_2025.pdf
9. Holm, S. (1979). A simple sequentially rejective multiple test procedure. *Scandinavian Journal of Statistics*, 6(2), 65–70.